In [1]:
import os
import pandas as pd
import numpy as np
from pathlib import Path
import nltk
nltk.download("punkt")
from nltk.tokenize import word_tokenize, sent_tokenize

data_folder = Path("data/train.csv")
data_path = os.path.join(os.getcwd(), data_folder)

print(data_path)

# # submission data format
# df = pd.read_parquet(data_path)
# df.head()

df = pd.read_csv(data_path)
df.head()

c:\Users\ps302\OneDrive\Desktop\Name_that_LLM\data\train.csv


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\ps302\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\ps302\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


,id,prompt_id,text,model
0,0018b3765972,p081,"When a language dies, a particular astronomy d...",qwen3.8-max
1,0041ac86ee93,p073,Data science communities matter because they a...,claude-sonnet-5
2,005a19083eae,p120,"In the final twilight of time, the last embers...",gemini-3.6-flash
3,009319596f10,p073,Data science communities matter because they t...,gpt-5.6-terra
4,00d3e8c92596,p097,"Deep inside each ear, three tiny loops of bone...",glm-5.3-flash


In [ ]:
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import FunctionTransformer
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, recall_score, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer



# 1. Train-test split (WITHOUT stratify to match your non-pipeline split)
df_train, df_test = train_test_split(df, test_size=0.2, random_state=42)

X_train = df_train[["text"]]
y_train = df_train["model"]
X_test = df_test[["text"]]
y_test = df_test["model"]

# 2. Label encode y
le = LabelEncoder()
y_train = le.fit_transform(y_train)
y_test = le.transform(y_test)

# text_feature (custom fucntion)
def text_features(X):
    X = np.asarray(X).ravel()

    return np.array([
        [ len(txt), len(word_tokenize(txt)), len(sent_tokenize(txt)) ] for txt in X
    ])

# 4. Combine both feature extractors into ONE ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ("tfidf", TfidfVectorizer(max_features=3000), "text"),
        ("num_features", FunctionTransformer(text_features, validate=False), "text")
    ]
)

# 5. Initialize and fit XGBoost
xgb = XGBClassifier(
    objective="multi:softprob",      # Multiclass classification
    n_estimators=100,
    learning_rate=0.1,
    max_depth=6,
    random_state=42,
    tree_method="hist",              # Fast histogram algorithm
    n_jobs=-1
)

# 7. Pipeline
pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("to_dense", FunctionTransformer(lambda x: x.toarray(), accept_sparse=True)),
    ("classifier", xgb),
])

pipeline

,steps,"[('preprocessor', ...), ('to_dense', ...), ...]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('tfidf', ...), ('num_features', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [3]:
# pipeline.named_steps
pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)

print(f"Accuracy Score: {accuracy_score(y_test, y_pred):.4f}")
print(f"Recall Score: {recall_score(y_test, y_pred, average='macro'):.4f}")

Accuracy Score: 0.1333
Recall Score: 0.1331


In [7]:
from sklearn.metrics import confusion_matrix
# print(confusion_matrix(y_test, y_pred))

In [8]:
test_df = pd.read_csv(r"C:\Users\ps302\OneDrive\Desktop\Name_that_LLM\data\test.csv")
test_df.head()



,id,text
0,00353fb1c541,"The more efficiently we use a resource, the mo..."
1,005f5a9a6139,"The Stern-Gerlach experiment, performed in 192..."
2,0062a53d5487,"Hundreds of millions of years ago, fluctuating..."


In [11]:
predictions = pipeline.predict(test_df[["text"]])
print(predictions)
print(le.inverse_transform(predictions)) 
predictions_f = le.inverse_transform(predictions)
# Submission file 
submission = pd.DataFrame({
    "id": test_df["id"],
    "model": predictions_f
})

submission.to_parquet("submission.parquet", index=False)
print(f"Saved submission.parquet with {len(submission)} rows.")
submission.head(10)

[11 17 10]
['gpt-5.5' 'qwen3.8-max' 'glm-5.3-flash']
Saved submission.parquet with 3 rows.


,id,model
0,00353fb1c541,gpt-5.5
1,005f5a9a6139,qwen3.8-max
2,0062a53d5487,glm-5.3-flash


In [ ]:
# from sklearn.pipeline import Pipeline, make_pipeline
# from sklearn.compose import ColumnTransformer
# from sklearn.preprocessing import FunctionTransformer
# from xgboost import XGBClassifier
# from sklearn.preprocessing import LabelEncoder
# from sklearn.metrics import accuracy_score, recall_score, confusion_matrix
# from sklearn.model_selection import train_test_split
# from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer



# # 1. Train-test split (WITHOUT stratify to match your non-pipeline split)
# df_train, df_test = train_test_split(df, test_size=0.2, random_state=42)

# X_train = df_train[["text"]]
# y_train = df_train["model"]
# X_test = df_test[["text"]]
# y_test = df_test["model"]

# # 2. Label encode y
# le = LabelEncoder()
# y_train = le.fit_transform(y_train)
# y_test = le.transform(y_test)

# # text_feature (custom fucntion)
# def text_features(X):
#     X = np.asarray(X).ravel()

#     return np.array([
#         [ len(txt), len(word_tokenize(txt)), len(sent_tokenize(txt)) ] for txt in X
#     ])

# # 4. Combine both feature extractors into ONE ColumnTransformer
# preprocessor = ColumnTransformer(
#     transformers=[
#         ("tfidf", TfidfVectorizer(max_features=3000), "text"),
#         ("num_features", FunctionTransformer(text_features, validate=False), "text")
#     ]
# )

# # 5. Initialize and fit XGBoost
# xgb = XGBClassifier(
#     objective="multi:softprob",      # Multiclass classification
#     n_estimators=100,
#     learning_rate=0.1,
#     max_depth=6,
#     random_state=42,
#     tree_method="hist",              # Fast histogram algorithm
#     n_jobs=-1
# )

# # 7. Pipeline
# pipeline = Pipeline([
#     ("preprocessor", preprocessor),
#     ("to_dense", FunctionTransformer(lambda x: x.toarray(), accept_sparse=True)),
#     ("classifier", xgb),
# ])

# # pipeline.named_steps
# pipeline.fit(X_train, y_train)
# y_pred = pipeline.predict(X_test)

# print(f"Accuracy Score: {accuracy_score(y_test, y_pred):.4f}")
# print(f"Recall Score: {recall_score(y_test, y_pred, average='macro'):.4f}")

Accuracy Score: 0.1685
Recall Score: 0.1773
